## Regression evaluation example

The following code shows a simple example of how to evaluate the predictions of a regression models.

We use the CASP.csv. The task is to predict the surface area of a protein. You can download the dataset in the [UCI machine learning repository](http://archive.ics.uci.edu/ml/datasets/Physicochemical+Properties+of+Protein+Tertiary+Structure).

In [1]:
import numpy as np

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import mean_squared_error, r2_score

from sklearn.linear_model import Lasso, Ridge
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

We read the dataset and split it into training and test parts. We use NumPy's utility function [`np.loadtxt`](https://docs.scipy.org/doc/numpy/reference/generated/numpy.loadtxt.html) to read the CSV file.
The first column of the data is the output that we want to predict, and the rest of the columns are the features. 

In [2]:
alldata = np.loadtxt('CASP.csv', skiprows=1, delimiter=',')

Yall = alldata[:,0]
Xall = alldata[:,1:]

Xtrain, Xtest, Ytrain, Ytest = train_test_split(Xall, Yall, test_size=0.2, random_state=0)

print(f'The size of the training set is {Xtrain.shape}.')

The size of the training set is (36584, 9).


We make a regression model. We don't need a vectorizer here, since the data is already in a numerical format.

You can play around and try a few different regressors. This is a complex nonlinear prediction problem, so a nonlinear model such as the [`RandomForestRegressor`](http://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) (this model will be discussed in the next lecture) works better than a linear regression model, e.g. [`Ridge`](http://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html). 

In [3]:
np.random.seed(0)

reg_model = RandomForestRegressor(n_estimators=100)
#reg_model = make_pipeline(StandardScaler(), Ridge())
#reg_model = DecisionTreeRegressor()
#reg_model = ExtraTreesRegressor()
#reg_model = GradientBoostingRegressor(n_estimators=500)
#reg_model = make_pipeline(StandardScaler(), MLPRegressor())

We train the regression model, and then evaluate the quality of the predictions by two metrics: [Mean Squared Error](https://en.wikipedia.org/wiki/Mean_squared_error) and $R^2$, the [coefficient of determination](https://en.wikipedia.org/wiki/Coefficient_of_determination).

The $R^2$ score has the advantage that it doesn't depend on the measurement scale: the score of a perfect regression model is always 1.0, while low-quality predictors have $R^2$ scores near 0. It seems our regressor is doing a fairly decent job in this case.

In [4]:
reg_model.fit(Xtrain, Ytrain)
Yguess = reg_model.predict(Xtest)

print('MSE =', mean_squared_error(Ytest, Yguess))
print('R2 =', r2_score(Ytest, Yguess))

MSE = 11.922856438324038
R2 = 0.6812101193242106


The following example shows how MSE and $R^2$ can be used in a cross-validation setup.

Note that the *negative* MSE will be used. The reason is that evaluation functions used in scikit-learn's cross-validation are assumed to return *high* values when the model is doing well.

In [5]:
print(cross_validate(reg_model, Xtrain, Ytrain, scoring='neg_mean_squared_error', return_train_score=False, cv=5))

print(cross_validate(reg_model, Xtrain, Ytrain, scoring='r2', return_train_score=False, cv=5))

{'fit_time': array([22.047616  , 21.91878462, 21.93965697, 22.09388328, 21.89536214]), 'score_time': array([0.1525979 , 0.1474123 , 0.14771914, 0.14954901, 0.14831781]), 'test_score': array([-13.11358543, -13.30059042, -12.85369581, -12.95424313,
       -12.99256824])}
{'fit_time': array([22.04086924, 21.97990108, 21.98864794, 21.965312  , 21.92915082]), 'score_time': array([0.14798069, 0.15391707, 0.14684582, 0.14575505, 0.14484787]), 'test_score': array([0.64539764, 0.64771894, 0.65454943, 0.6548167 , 0.65622705])}
